In [21]:
# ── STEP 1 · Import Libraries ─────────────────
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    ExtraTreesClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from lightgbm import LGBMClassifier

# ── STEP 2 · Load Data ────────────────────────
df = pd.read_csv("german_credit_data.csv")

# Fill missing categorical columns with 'unknown'
df["Saving accounts"].fillna("unknown", inplace=True)
df["Checking account"].fillna("unknown", inplace=True)


df = pd.read_csv("german_credit_data.csv")
X_raw = df.drop(["Risk", "Index"], axis=1)
cat_cols = X_raw.select_dtypes(include=['object', 'category']).columns

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(X_raw[cat_cols])

encoded_cats = encoder.transform(X_raw[cat_cols])
encoded_df = pd.DataFrame(encoded_cats, columns=encoder.get_feature_names_out(cat_cols))

X = pd.concat([X_raw.drop(cat_cols, axis=1).reset_index(drop=True), encoded_df], axis=1)
y = df["Risk"]

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# ── STEP 3 · Train / Test Split ───────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# ── STEP 4 · Define Models + Param Grids ──────
models = {

    "Logistic Regression": {
        "model": LogisticRegression(max_iter=1000, random_state=42),
        "params": {
            "C":        [0.01, 0.1, 1, 10, 100],
            "penalty":  ["l1", "l2"],
            "solver":   ["liblinear", "saga"],
        }
    },

    "Random Forest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            "n_estimators":      [100, 200, 300],
            "max_depth":         [None, 5, 10, 20],
            "min_samples_split": [2, 5, 10],
            "min_samples_leaf":  [1, 2, 4],
            "max_features":      ["sqrt", "log2"],
        }
    },

    "Gradient Boosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            "n_estimators":      [100, 200, 300],
            "learning_rate":     [0.01, 0.05, 0.1, 0.2],
            "max_depth":         [3, 5, 7],
            "subsample":         [0.7, 0.8, 1.0],
            "min_samples_split": [2, 5],
        }
    },

    "Extra Trees": {
        "model": ExtraTreesClassifier(random_state=42),
        "params": {
            "n_estimators":      [100, 200, 300],
            "max_depth":         [None, 5, 10],
            "min_samples_split": [2, 5, 10],
            "max_features":      ["sqrt", "log2"],
        }
    },

    "SVM": {
        "model": SVC(probability=True, random_state=42),
        "params": {
            "C":      [0.1, 1, 10, 100],
            "kernel": ["rbf", "linear", "poly"],
            "gamma":  ["scale", "auto"],
        }
    },

    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            "n_neighbors": [3, 5, 7, 9, 11],
            "weights":     ["uniform", "distance"],
            "metric":      ["euclidean", "manhattan"],
        }
    },
}

# ── STEP 5 · Run GridSearchCV for All Models ──
results = {}

for name, config in models.items():
    print(f"\n🔍 Tuning: {name} ...")

    grid = GridSearchCV(
        estimator  = config["model"],
        param_grid = config["params"],
        cv         = cv,
        scoring    = "accuracy",
        n_jobs     = -1,
        verbose    = 0,
    )

    grid.fit(X_train, y_train)

    best    = grid.best_estimator_
    y_pred  = best.predict(X_test)
    y_prob  = best.predict_proba(X_test)[:,1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    results[name] = {
        "best_params": grid.best_params_,
        "cv_accuracy": grid.best_score_,
        "test_accuracy": acc,
        "test_auc": auc,
        "model": best,
        "y_pred": y_pred,
    }

    print(f"   Best params  : {grid.best_params_}")
    print(f"   CV  Accuracy : {grid.best_score_:.4f}")
    print(f"   Test Accuracy: {acc:.4f}  |  AUC: {auc:.4f}")

# ── STEP 6 · Compare All Models ───────────────
print("\n" + "="*60)
print("MODEL COMPARISON")
print("="*60)

summary = pd.DataFrame({
    name: {
        "CV Accuracy":   f"{r['cv_accuracy']:.4f}",
        "Test Accuracy": f"{r['test_accuracy']:.4f}",
        "Test AUC":      f"{r['test_auc']:.4f}",
    }
    for name, r in results.items()
}).T.sort_values("Test Accuracy", ascending=False)

print(summary.to_string())

# ── STEP 7 · Best Model Full Report ───────────
best_name = summary.index[0]
best_result = results[best_name]

print(f"\n🏆 Best Model: {best_name}")
print(f"   Best Params: {best_result['best_params']}")
print(f"\n{classification_report(y_test, best_result['y_pred'], target_names=['Bad', 'Good'])}")

C:\Users\Aditya_Bansal\AppData\Local\Temp\ipykernel_10676\2462792258.py:21: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["Saving accounts"].fillna("unknown", inplace=True)
C:\Users\Aditya_Bansal\AppData\Local\Temp\ipykernel_10676\2462792258.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always beha


🔍 Tuning: Logistic Regression ...
   Best params  : {'C': 0.1, 'penalty': 'l1', 'solver': 'liblinear'}
   CV  Accuracy : 0.7300
   Test Accuracy: 0.7350  |  AUC: 0.7648

🔍 Tuning: Random Forest ...
   Best params  : {'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 10, 'n_estimators': 200}
   CV  Accuracy : 0.7475
   Test Accuracy: 0.7800  |  AUC: 0.7749

🔍 Tuning: Gradient Boosting ...
   Best params  : {'learning_rate': 0.01, 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 200, 'subsample': 0.7}
   CV  Accuracy : 0.7637
   Test Accuracy: 0.7850  |  AUC: 0.7643

🔍 Tuning: Extra Trees ...
   Best params  : {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 10, 'n_estimators': 100}
   CV  Accuracy : 0.7337
   Test Accuracy: 0.7400  |  AUC: 0.7481

🔍 Tuning: SVM ...
   Best params  : {'C': 1, 'gamma': 'scale', 'kernel': 'linear'}
   CV  Accuracy : 0.7288
   Test Accuracy: 0.7250  |  AUC: 0.7324

🔍 Tuning: KNN ...
   Best params  

In [22]:
X_df = pd.DataFrame(data=X,columns=X.columns)
y_df = pd.DataFrame(data=y,columns=["Risk"])
y_df['Risk'] = [1 if x=="good" else 0 for x in y_df["Risk"]]
merged_df = pd.concat([X_df, y_df], axis=1)

In [23]:
merged_df

,Age,Job,Credit amount,Duration,Sex_female,Sex_male,Housing_free,Housing_own,Housing_rent,Saving accounts_little,...,Checking account_nan,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others,Risk
0,2.766456,0.146949,-0.745131,-1.236478,-0.670280,0.670280,-0.347960,0.634448,-0.466933,-1.232433,...,-0.806328,-0.327749,-0.712949,-0.110208,-0.250398,-0.470108,1.603567,-0.149983,-0.110208,1
1,-1.191404,0.146949,0.949817,2.248194,1.491914,-1.491914,-0.347960,0.634448,-0.466933,0.811403,...,-0.806328,-0.327749,-0.712949,-0.110208,-0.250398,-0.470108,1.603567,-0.149983,-0.110208,0
2,1.183312,-1.383771,-0.416562,-0.738668,-0.670280,0.670280,-0.347960,0.634448,-0.466933,0.811403,...,1.240190,-0.327749,-0.712949,-0.110208,3.993639,-0.470108,-0.623610,-0.149983,-0.110208,1
3,0.831502,0.146949,1.634247,1.750384,-0.670280,0.670280,2.873893,-1.576173,-0.466933,0.811403,...,-0.806328,-0.327749,-0.712949,-0.110208,-0.250398,2.127172,-0.623610,-0.149983,-0.110208,1
4,1.535122,0.146949,0.566664,0.256953,-0.670280,0.670280,2.873893,-1.576173,-0.466933,0.811403,...,-0.806328,-0.327749,1.402626,-0.110208,-0.250398,-0.470108,-0.623610,-0.149983,-0.110208,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,-0.399832,-1.383771,-0.544162,-0.738668,1.491914,-1.491914,-0.347960,0.634448,-0.466933,0.811403,...,1.240190,-0.327749,-0.712949,-0.110208,-0.250398,2.127172,-0.623610,-0.149983,-0.110208,1
996,0.391740,1.677670,0.207612,0.754763,-0.670280,0.670280,-0.347960,0.634448,-0.466933,0.811403,...,-0.806328,-0.327749,1.402626,-0.110208,-0.250398,-0.470108,-0.623610,-0.149983,-0.110208,1
997,0.215835,0.146949,-0.874503,-0.738668,-0.670280,0.670280,-0.347960,0.634448,-0.466933,0.811403,...,1.240190,-0.327749,-0.712949,-0.110208,-0.250398,-0.470108,1.603567,-0.149983,-0.110208,1
998,-1.103451,0.146949,-0.505528,1.999289,-0.670280,0.670280,2.873893,-1.576173,-0.466933,0.811403,...,-0.806328,-0.327749,-0.712949,-0.110208,-0.250398,-0.470108,1.603567,-0.149983,-0.110208,0


In [24]:
import joblib
import pickle

# Save CSV
merged_df.to_csv("merged_data.csv", index=False)

# Save Scaler
joblib.dump(scaler, "scaler.pkl")

# Save Model
joblib.dump(best, "model.pkl")

# Save Encoding
with open('encoder.pkl', 'wb') as f:
    pickle.dump(encoder, f) 

In [27]:
# Example user input
user_input = {
    'Age': 25,
    'Sex': 'male',
    'Job': 2,
    'Housing': 'own',
    'Saving accounts': 'little',
    'Checking account': 'moderate',
    'Credit amount': 1500,
    'Duration': 12,
    'Purpose': 'radio/TV'
}

# 1. Convert input to DataFrame
input_df = pd.DataFrame([user_input])

# 2. Load the encoder
scaler = joblib.load('scaler.pkl')
model = joblib.load('model.pkl')
with open('encoder.pkl', 'rb') as f:
    loaded_encoder = pickle.load(f)

# 3. Transform the user input
cat_cols = loaded_encoder.feature_names_in_ # Or the list you used during training
encoded_cats = loaded_encoder.transform(input_df[cat_cols])
encoded_df = pd.DataFrame(encoded_cats, columns=loaded_encoder.get_feature_names_out(cat_cols))

# 4. Reconstruct the final vector for the model
encoded_input = pd.concat([input_df.drop(cat_cols, axis=1).reset_index(drop=True), encoded_df], axis=1)
scaled_input = scaler.transform(encoded_input)
# Now you can safely call model.predict(final_input)

In [28]:
scaled_input

array([[-0.92754658,  0.14694918, -0.62781066, -0.73866754, -0.67028006,
         0.67028006, -0.3479601 ,  0.63444822, -0.4669334 ,  0.81140298,
        -0.33886163, -0.25929878, -0.22454436, -0.47327604, -0.61433742,
         1.6484757 , -0.25929878, -0.80632811, -0.32774947, -0.71294854,
        -0.11020775, -0.2503982 , -0.47010767,  1.60356745, -0.14998296,
        -0.11020775]])

In [31]:
prediction = model.predict(scaled_input)

c:\solution_challenge\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but KNeighborsClassifier was fitted with feature names
  warnings.warn(


In [32]:
prediction[0]

'good'